# Data Preparation


Aufbereitung des Master-Datensatzes für die Modellierung.

## Agenda


Aus `01_exploration.ipynb` — Befunde bestimmen was in welcher Phase passiert.

| Topic | Exploration | Preparation  |
|:---|:---|:---|
| **— Cleaning ——————————** | | | 
| Delay | Extreme Werte ±8.3h — physikalisch nicht plausibel | Filter `\|delay\| > 3.600s` |
| Delay | 74.669 Zeilen: Schedule vorhanden, kein Delay | Herausfiltern |
| BPUIC | 0.10% anomale IDs > 100.000.000 | Herausfiltern | 
| Meteo | `humidity` > 100% — Sensor-Drift | `.clip(0, 100)` | 
| Datenqualität | 1.72% Duplikate | `unique()` | 
| Events | 78.5% null — kein Event ist Normalfall | `null` → `"no_event"` |
| District | 6.87% null — Haltestellen außerhalb Stadtgebiet | `null` → `"outside"` | 
| **— Split —————————————** | | | 
| Split | Zeitreihendaten — kein Random Shuffle | 2023+2024 → Train / 2025 → Test (65.6% / 34.4%) | 
| **— Preprocessing ————————** | | | 
| Meteo | Stündliche Messausfälle (~0.14–0.35%), zeitlich klumpend | Forward/Backward Fill |
| **— Feature Engineering ———** | | | 
| Meteo | `precipitation` zero-inflated | `has_rain` Flag + Rohwert behalten | 
| Meteo | Wetter→Delay: r max 0.03 linear | Schwellenwert-Flags statt Rohwerte | 
| Delay | Skewness 38–43 — non-linear, long tail | Keine Bereinigung — XGBoost robust | 
| **— Export ————————————** | | | 
| Output | Feature-Files für Modellierung | `train_features.parquet` · `test_features.parquet` | 

> **Tatsächlich entfernt in Cleaning:** ~2.2 Mio. Zeilen (~2.3%) · verbleibend: **~92.2 Mio.**

> **Leakage-Prinzip:** Alles was Parameter aus den Daten lernt kommt nach dem Split — gefittet auf Train, angewendet auf Test.

## Setup



### Imports

In [ ]:
from zh_tram_flow.notebook import *
from zh_tram_flow.data.loader       import load_raw, split_by_year
from zh_tram_flow.data.cleaning     import run_cleaning
from zh_tram_flow.data.split        import temporal_split
from zh_tram_flow.data.preprocessing import run_preprocessing
from zh_tram_flow.data.export       import run_export

INTERIM   = PATHS["interim"]
PROCESSED = PATHS["processed"]
INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

YEARS = ["2023", "2024", "2025"]

%load_ext autoreload
%autoreload 2

### Pfade und Datensätze

Pfade und Konfiguration kommen aus `zh_tram_flow.config`.  
Der Master-Datensatz wird als LazyFrame geladen — kein RAM-Verbrauch bis zur ersten Operation.  
Für eine bessere Handhabung der großen Datenmenge werden zusätzlich Jahres-Pakete erstellt.

In [ ]:
section_header("Load Raw Data")
# → zh_tram_flow/data/loader.py

lf_raw     = load_raw()
lazyframes = split_by_year(lf_raw, YEARS, INTERIM)

## Cleaning


*Fehler entfernen auf Basis von Domainwissen — vor dem Split, kein Leakage-Risiko.*

Nur Regeln die unabhängig von Statistiken gelten:

| Schritt | Was | Warum |
|:---|:---|:---|
| Duplikate | Vollständig doppelte Zeilen entfernen | GTFS-Join-Artefakt |
| BPUIC | Anomale Haltestellen-IDs entfernen | Außerhalb VBZ-Bereich |
| Delay-Mismatch | Schedule ohne Delay entfernen | Übertragungsfehler |
| Extreme Delays | `\|delay\| > 3.600s` entfernen | Physikalisch nicht plausibel |
| Humidity | Über 100% kappen | Sensor-Kalibrierungsdrift |
| Null-Kategorien | District und Events mit Label füllen | Kein Fehler — definierter Zustand |

In [ ]:
section_header("Structural Cleaning")
# → zh_tram_flow/data/cleaning.py

clean_files = run_cleaning(lazyframes, YEARS, INTERIM)

## Split


*Temporal aufteilen — kein Random Shuffle.*

Zeitreihendaten dürfen nicht zufällig gesplittet werden: das Modell würde sonst auf Daten
trainieren die zeitlich nach dem Test liegen — es "kennt die Zukunft".

| Set | Zeitraum | Zeilen (nach Cleaning) |
|:---|:---|:---|
| Train | 2023 + 2024 | → aus Cleaning-Output |
| Test | 2025 | → aus Cleaning-Output |

> Konkretes Verhältnis: siehe Ausgabe unten — berechnet aus den bereinigten Jahresdateien.  
> Test-Daten werden bis zur finalen Evaluation nicht angefasst.

In [ ]:
section_header("Train / Test Split")
# → zh_tram_flow/data/split.py

out_train, out_test = temporal_split(clean_files, INTERIM)

## Preprocessing


*Daten modellbereit machen — nach dem Split, Parameter nur aus Train.*

| Schritt | Bedeutung | Status |
|:---|:---|:---|
| **Imputation** | Fehlende Werte füllen | ✅ Meteo Forward/Backward Fill |
| **Scaling** | Wertebereiche normalisieren | → Modeling-Notebook |
| **Encoding** | Kategorien in Zahlen umwandeln | → Modeling-Notebook |
| **Outlier Handling** | Ausreißer behandeln | → Modeling-Notebook |

> Alles was Parameter aus den Daten lernt: erst auf Train fitten, dann auf Test anwenden.

In [ ]:
section_header("Meteo Imputation")
# → zh_tram_flow/data/preprocessing.py

out_train_prep, out_test_prep = run_preprocessing(out_train, out_test, PROCESSED)

## Feature Engineering


*Neue Spalten aus vorhandenen ableiten — nach dem Split.*

| Kategorie | Features |
|:---|:---|
| Zeit | `hour` · `weekday` · `month` · `season` · `is_weekend` · `is_rush_hour` |
| Netz | `gtfs_year` — struktureller Fahrplanwechsel Dez 2023 (j23 vs. j24/j25) |
| Wetter | `has_rain` · `has_heavy_rain` · `is_windy` · `has_snow` · `has_flood` |
| Event | `is_holiday` · `has_event` · `event_weight` |
| Ausfall | `is_canceled` |

> `trip_id` und `stop_sequence` sind im Master-Datensatz verfügbar (seit F-TARGET-08) — nicht als direkte Modell-Features, sondern als Schlüssel für Trip-Level-Analysen (Kaskaden, Hotspots) in `03_analysis_network.ipynb`.

> Encoding-Parameter (Target-Encoding für Linie, Stadtkreis etc.) werden im Modeling-Notebook auf Train gefittet.

In [ ]:
section_header("Feature Engineering")
# → zh_tram_flow/features/temporal.py   add_time_features
# → zh_tram_flow/features/weather.py    add_weather_flags
# → zh_tram_flow/features/events.py     add_event_features
# → zh_tram_flow/features/delays.py     add_delay_features
# → inline: gtfs_year  (operating_date < 2024-01-01 → "j23", sonst "j24_j25")
#            trip_id / stop_sequence: bereits im Master verfügbar, kein Engineering nötig

## Export


In [ ]:
section_header("Export")
# → zh_tram_flow/data/export.py

out_train_feat, out_test_feat = run_export(out_train_prep, out_test_prep, PROCESSED)

In [ ]:
lf_train_feat = pl.scan_parquet(out_train_feat)
lf_test_feat  = pl.scan_parquet(out_test_feat)

print(lf_train_feat.collect_schema().names())
print()
print(lf_test_feat.collect_schema().names())

#### Cleaning — Ergebnis

| Jahr | Roh | Nach Cleaning | Entfernt | Set |
|:---|---:|---:|---:|:---|
| 2023 | 31,765,675 | 30,345,625 | 1,420,050 (4.5%) | Train |
| 2024 | 30,773,025 | 30,153,216 | 619,809 (2.0%) | Train |
| 2025 | 31,892,729 | 31,719,263 | 173,466 (0.5%) | Test |
| **Total** | **94,431,429** | **92,218,104** | **2,213,325 (2.3%)** | |

#### Feature Files

|           | Train      | Test       |
|:---       |---:        |---:        |
| Zeilen    | 60,498,841 | 31,719,263 |
| Spalten   | 40         | 40         |
| Größe     | 571 MB     | 298 MB     |

26 original columns + 15 new features — `hour` · `weekday` · `month` · `season` · `is_weekend` · `is_rush_hour` · `has_rain` · `has_heavy_rain` · `is_windy` · `has_snow` · `has_flood` · `is_holiday` · `has_event` · `event_weight` · `is_canceled`

> Master-Datensatz seit F-TARGET-08: 26 Spalten (+`trip_id`, +`stop_sequence`). `gtfs_year` kommt als 41. Spalte nach dem nächsten Feature-Engineering-Run hinzu.